<a href="https://colab.research.google.com/github/emrizky1/machine-learning/blob/main/JS09/JS09_Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 16.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 37.3 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=2aeaae0b4fe24b2832fe78df0a38cf7a894d56f23b9737dc5c3270102ceb66d2
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679794 sha256=d04c6df9e99c7a009b8bb42d534e7688985f9555d859f0b6e5438235ef97abc8
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f5

In [2]:
import time
import numpy as np
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex

np.random.seed(42)

K = 10
N_QUERIES = 1000
SIZES = [1_000, 1_000_000]
DIMS = [2, 5]
METRICS = ["euclidean", "cosine", "inner_product"]

In [3]:
def ground_truth(X, Q, metric, k):
    """Exact NN (brute force) as the reference for recall."""
    d = X.shape[1]
    if metric == "euclidean":
        index = faiss.IndexFlatL2(d); Xg, Qg = X, Q
    elif metric == "cosine":
        index = faiss.IndexFlatIP(d)
        Xg, Qg = X.copy(), Q.copy()
        faiss.normalize_L2(Xg); faiss.normalize_L2(Qg)
    else:
        index = faiss.IndexFlatIP(d); Xg, Qg = X, Q
    index.add(Xg)
    _, I = index.search(Qg, k)
    return I

def recall_at_k(I_approx, I_true):
    return np.mean([len(set(a) & set(t)) / len(t) for a, t in zip(I_approx, I_true)])

def run_annoy(X, Q, metric, k, n_trees=10):
    if metric == "inner_product":
        return None
    d = X.shape[1]
    idx = AnnoyIndex(d, "euclidean" if metric == "euclidean" else "angular")
    t0 = time.perf_counter()
    for i, v in enumerate(X):
        idx.add_item(i, v)
    idx.build(n_trees)
    build = time.perf_counter() - t0

    t0 = time.perf_counter()
    I = np.array([idx.get_nns_by_vector(q, k) for q in Q])
    query = (time.perf_counter() - t0) / len(Q) * 1000
    return I, build, query

def run_faiss(X, Q, metric, k):
    n, d = X.shape
    nlist = max(4, int(np.sqrt(n)))
    m = d
    Xf, Qf = X.copy(), Q.copy()
    if metric == "cosine":
        faiss.normalize_L2(Xf); faiss.normalize_L2(Qf)
    if metric == "euclidean":
        quantizer = faiss.IndexFlatL2(d); mt = faiss.METRIC_L2
    else:
        quantizer = faiss.IndexFlatIP(d); mt = faiss.METRIC_INNER_PRODUCT

    index = faiss.IndexIVFPQ(quantizer, d, nlist, m, 8, mt)
    index.nprobe = 10

    t0 = time.perf_counter()
    train_set = Xf if n <= 100_000 else Xf[np.random.choice(n, 100_000, replace=False)]
    index.train(train_set)
    index.add(Xf)
    build = time.perf_counter() - t0

    t0 = time.perf_counter()
    _, I = index.search(Qf, k)
    query = (time.perf_counter() - t0) / len(Qf) * 1000
    return I, build, query

def run_hnsw(X, Q, metric, k, M=16, ef_construction=100, ef_search=50):
    n, d = X.shape
    space = {"euclidean": "l2", "cosine": "cosine", "inner_product": "ip"}[metric]
    idx = hnswlib.Index(space=space, dim=d)
    idx.init_index(max_elements=n, ef_construction=ef_construction, M=M)

    t0 = time.perf_counter()
    idx.add_items(X, np.arange(n))
    build = time.perf_counter() - t0

    idx.set_ef(ef_search)
    t0 = time.perf_counter()
    I, _ = idx.knn_query(Q, k=k)
    query = (time.perf_counter() - t0) / len(Q) * 1000
    return I, build, query

In [4]:
results = []

for n in SIZES:
    for d in DIMS:
        print(f"=== {n:,} data / {d}D ===")
        X = (np.random.rand(n, d) * 100).astype("float32")
        Q = (np.random.rand(N_QUERIES, d) * 100).astype("float32")

        for metric in METRICS:
            I_true = ground_truth(X, Q, metric, K)

            for name, fn in [("ANNOY", run_annoy), ("FAISS", run_faiss), ("HNSW", run_hnsw)]:
                out = fn(X, Q, metric, K)
                if out is None:
                    results.append(dict(data=f"{n:,}/{d}D", metric=metric, algo=name,
                                        build_s=np.nan, query_ms=np.nan, recall=np.nan))
                    continue
                I, build, query = out
                rec = recall_at_k(I, I_true)
                results.append(dict(data=f"{n:,}/{d}D", metric=metric, algo=name,
                                    build_s=build, query_ms=query, recall=rec))
                print(f"  {metric:14s} {name:6s} build={build:8.3f}s  "
                      f"query={query:.4f}ms  recall@{K}={rec:.3f}")

df = pd.DataFrame(results)

=== 1,000 data / 2D ===
  euclidean      ANNOY  build=   0.028s  query=0.0314ms  recall@10=0.994
  euclidean      FAISS  build=   0.031s  query=0.0053ms  recall@10=0.994
  euclidean      HNSW   build=   0.026s  query=0.0094ms  recall@10=1.000
  cosine         ANNOY  build=   0.034s  query=0.0228ms  recall@10=0.995
  cosine         FAISS  build=   0.032s  query=0.0050ms  recall@10=0.726
  cosine         HNSW   build=   0.022s  query=0.0077ms  recall@10=0.999
  inner_product  FAISS  build=   0.028s  query=0.0056ms  recall@10=0.450
  inner_product  HNSW   build=   0.029s  query=0.0087ms  recall@10=0.991
=== 1,000 data / 5D ===
  euclidean      ANNOY  build=   0.017s  query=0.0218ms  recall@10=0.966
  euclidean      FAISS  build=   0.080s  query=0.0071ms  recall@10=0.991
  euclidean      HNSW   build=   0.031s  query=0.0132ms  recall@10=1.000
  cosine         ANNOY  build=   0.022s  query=0.0242ms  recall@10=0.980
  cosine         FAISS  build=   0.081s  query=0.0061ms  recall@10=0.966
  c

In [5]:
order = ["1,000/2D", "1,000/5D", "1,000,000/2D", "1,000,000/5D"]

def make_table(metric, value, fmt):
    t = (df[df.metric == metric]
         .pivot(index="data", columns="algo", values=value)
         .reindex(order)[["ANNOY", "FAISS", "HNSW"]])
    t.index.name = "Number of Data/Dimensions"
    return t.map(lambda v: "N/A" if pd.isna(v) else fmt.format(v))

for metric in METRICS:
    print(f"\n########## Metric: {metric} ##########")
    for value, title, fmt in [("query_ms", "Avg query time (ms)", "{:.4f}"),
                              ("build_s",  "Build time (s)",     "{:.3f}"),
                              ("recall",   f"Recall@{K}",        "{:.3f}")]:
        print(f"\n{title}")
        display(make_table(metric, value, fmt))


########## Metric: euclidean ##########

Avg query time (ms)


algo,ANNOY,FAISS,HNSW
Number of Data/Dimensions,,,
"1,000/2D",0.0314,0.0053,0.0094
"1,000/5D",0.0218,0.0071,0.0132
"1,000,000/2D",0.0510,0.0712,0.0195
"1,000,000/5D",0.0541,0.0912,0.0330



Build time (s)


algo,ANNOY,FAISS,HNSW
Number of Data/Dimensions,,,
"1,000/2D",0.028,0.031,0.026
"1,000/5D",0.017,0.080,0.031
"1,000,000/2D",26.614,11.657,53.961
"1,000,000/5D",20.095,24.112,84.657



Recall@10


algo,ANNOY,FAISS,HNSW
Number of Data/Dimensions,,,
"1,000/2D",0.994,0.994,1.000
"1,000/5D",0.966,0.991,1.000
"1,000,000/2D",0.996,0.965,1.000
"1,000,000/5D",0.946,0.976,1.000


... (8 outputs left out)


Build time (s)


algo,ANNOY,FAISS,HNSW
Number of Data/Dimensions,,,
"1,000/2D",N/A,0.028,0.029
"1,000/5D",N/A,0.082,0.036
"1,000,000/2D",N/A,10.994,36.996
"1,000,000/5D",N/A,25.753,66.117



Recall@10


algo,ANNOY,FAISS,HNSW
Number of Data/Dimensions,,,
"1,000/2D",N/A,0.450,0.991
"1,000/5D",N/A,0.903,0.979
"1,000,000/2D",N/A,0.003,0.963
"1,000,000/5D",N/A,0.025,0.701


In [6]:
def combined(metric):
    d = df[df.metric == metric].copy()
    d["cell"] = d.apply(lambda r: "N/A" if pd.isna(r.recall) else
                        f"{r.query_ms:.4f} ms | R={r.recall:.2f} | build {r.build_s:.2f}s", axis=1)
    t = d.pivot(index="data", columns="algo", values="cell").reindex(order)[["ANNOY", "FAISS", "HNSW"]]
    t.index.name = "Number of Data/Dimensions"
    return t

for metric in METRICS:
    print(f"\nMetric: {metric}")
    display(combined(metric))


Metric: euclidean


algo,ANNOY,FAISS,HNSW
Number of Data/Dimensions,,,
"1,000/2D",0.0314 ms | R=0.99 | build 0.03s,0.0053 ms | R=0.99 | build 0.03s,0.0094 ms | R=1.00 | build 0.03s
"1,000/5D",0.0218 ms | R=0.97 | build 0.02s,0.0071 ms | R=0.99 | build 0.08s,0.0132 ms | R=1.00 | build 0.03s
"1,000,000/2D",0.0510 ms | R=1.00 | build 26.61s,0.0712 ms | R=0.96 | build 11.66s,0.0195 ms | R=1.00 | build 53.96s
"1,000,000/5D",0.0541 ms | R=0.95 | build 20.09s,0.0912 ms | R=0.98 | build 24.11s,0.0330 ms | R=1.00 | build 84.66s



Metric: cosine


algo,ANNOY,FAISS,HNSW
Number of Data/Dimensions,,,
"1,000/2D",0.0228 ms | R=1.00 | build 0.03s,0.0050 ms | R=0.73 | build 0.03s,0.0077 ms | R=1.00 | build 0.02s
"1,000/5D",0.0242 ms | R=0.98 | build 0.02s,0.0061 ms | R=0.97 | build 0.08s,0.0127 ms | R=1.00 | build 0.03s
"1,000,000/2D",0.0602 ms | R=0.07 | build 94.82s,0.0809 ms | R=0.01 | build 11.91s,0.0314 ms | R=0.31 | build 86.33s
"1,000,000/5D",0.0539 ms | R=0.97 | build 27.74s,0.0894 ms | R=0.60 | build 25.39s,0.0684 ms | R=1.00 | build 77.76s



Metric: inner_product


algo,ANNOY,FAISS,HNSW
Number of Data/Dimensions,,,
"1,000/2D",N/A,0.0056 ms | R=0.45 | build 0.03s,0.0087 ms | R=0.99 | build 0.03s
"1,000/5D",N/A,0.0066 ms | R=0.90 | build 0.08s,0.0153 ms | R=0.98 | build 0.04s
"1,000,000/2D",N/A,0.1617 ms | R=0.00 | build 10.99s,0.0086 ms | R=0.96 | build 37.00s
"1,000,000/5D",N/A,0.0907 ms | R=0.02 | build 25.75s,0.0261 ms | R=0.70 | build 66.12s
